In [15]:
import numpy as np
from tqdm import tqdm, trange

from reader import *
from encoder import *

bunny_model = Reader.read_from_file('assets/bunny.obj')

baseline_encoder = BaselineEncoder()
baseline_compressed_model = baseline_encoder.encode(bunny_model)

import matplotlib.pyplot as plt


def print_statistics(compressed_model, name):
    print(f"{name} statistics:")
    print(f"- Bytes per triangle: {compressed_model.bits_per_triangle / 8:.2f}")
    print(f"- Bytes per vertex: {compressed_model.bits_per_vertex / 8:.2f}")
    print(f"- Compression rate: {baseline_compressed_model.bits_per_vertex / compressed_model.bits_per_vertex:.2f}")


print_statistics(baseline_compressed_model, "BaselineEncoder")

BaselineEncoder statistics:
- Bytes per triangle: 18.05
- Bytes per vertex: 35.82
- Compression rate: 1.00


In [96]:
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Set, Optional
from collections import deque, namedtuple


@dataclass
class TriangleMesh:
    vertices: List[Vertex]
    triangles: List[Triangle]
    tri_to_edges: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_tris: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_vertices: Dict[int, Tuple[int, int]] = field(default_factory=dict)

    def build_adjacency(self) -> None:
        """
        Populate:
        - tri_to_edges: map triangle index to its three mesh-edge IDs
        - edge_to_tris: map mesh-edge ID to list of triangles sharing that edge
        - edge_to_vertices: map mesh-edge ID to the vertex index pair
        """
        self.tri_to_edges.clear()
        self.edge_to_tris.clear()
        self.edge_to_vertices.clear()
        edge_map: Dict[Tuple[int, int], int] = {}
        next_eid = 0
        for tidx, tri in enumerate(self.triangles):
            eids: List[int] = []
            for u, v in ((tri.a, tri.b), (tri.b, tri.c), (tri.c, tri.a)):
                key = (min(u, v), max(u, v))
                if key not in edge_map:
                    edge_map[key] = next_eid
                    self.edge_to_tris[next_eid] = []
                    self.edge_to_vertices[next_eid] = key
                    next_eid += 1
                eid = edge_map[key]
                eids.append(eid)
                self.edge_to_tris[eid].append(tidx)
            self.tri_to_edges[tidx] = eids

# -------------------------------------------------------------------
# 3. Dual Graph Representation
# -------------------------------------------------------------------
@dataclass
class DualGraph:
    num_triangles: int
    adj: Dict[int, List[Tuple[int, int]]] = field(default_factory=dict)
    is_strip_edge: Dict[int, bool] = field(default_factory=dict)

    @classmethod
    def from_mesh(cls, mesh: TriangleMesh) -> 'DualGraph':
        dg = cls(num_triangles=len(mesh.triangles))
        dg.adj = {t: [] for t in range(dg.num_triangles)}
        for eid, tris in mesh.edge_to_tris.items():
            dg.is_strip_edge[eid] = False
            if len(tris) < 2:
                continue
            for i in range(len(tris)):
                for j in range(i + 1, len(tris)):
                    t0, t1 = tris[i], tris[j]
                    dg.adj[t0].append((t1, eid))
                    dg.adj[t1].append((t0, eid))
        return dg

    def mark_strip_edge(self, eid: int) -> None:
        self.is_strip_edge[eid] = True

    def unmark_strip_edge(self, eid: int) -> None:
        self.is_strip_edge[eid] = False

    def incident_strip_edge_count(self, tri: int) -> int:
        return sum(self.is_strip_edge[e] for _, e in self.adj[tri])

# -------------------------------------------------------------------
# 4. Tunnel Finder
# -------------------------------------------------------------------
BFSState = namedtuple('BFSState', ['tri', 'parity', 'prev_eid', 'prev_state'])

class TunnelFinder:
    def __init__(self, graph: DualGraph):
        self.graph = graph

    def find_tunnel(self, start: int, max_depth: int = 100) -> Optional[List[int]]:
        visited = set()
        queue = deque([BFSState(start, 0, None, None)])
        while queue:
            state = queue.popleft()
            path = self._reconstruct(state)
            if len(path) > max_depth:
                continue
            for nbr, eid in self.graph.adj[state.tri]:
                want = (state.parity == 1)
                if self.graph.is_strip_edge[eid] != want:
                    continue
                nxt = BFSState(nbr, 1 - state.parity, eid, state)
                key = (nbr, nxt.parity)
                if key in visited:
                    continue
                visited.add(key)
                if nxt.parity == 1 and self._is_terminal(nbr) and nbr != start:
                    return self._reconstruct(nxt)
                queue.append(nxt)
        return None

    def _reconstruct(self, state: BFSState) -> List[int]:
        path = []
        while state.prev_eid is not None:
            path.append(state.prev_eid)
            state = state.prev_state
        return list(reversed(path))

    def _is_terminal(self, tri: int) -> bool:
        return self.graph.incident_strip_edge_count(tri) <= 1

# -------------------------------------------------------------------
# 5. Stripifier Control
# -------------------------------------------------------------------
class Stripifier:
    def __init__(self, mesh: TriangleMesh):
        self.mesh = mesh
        self.mesh.build_adjacency()
        self.graph = DualGraph.from_mesh(mesh)
        self._init()
        self.tfinder = TunnelFinder(self.graph)
        self.terminals = {t for t in range(self.graph.num_triangles)
                          if self.graph.incident_strip_edge_count(t) <= 1}

    def _init(self) -> None:
        for eid in self.graph.is_strip_edge:
            self.graph.unmark_strip_edge(eid)

    def run(self, iters: int = 1000) -> None:
        for _ in range(iters):
            moved = False
            for st in list(self.terminals):
                tun = self.tfinder.find_tunnel(st)
                if tun:
                    self._apply(tun)
                    moved = True
                    break
            if not moved:
                break

    def _apply(self, path: List[int]) -> None:
        affected = set()
        for eid in path:
            if self.graph.is_strip_edge[eid]:
                self.graph.unmark_strip_edge(eid)
            else:
                self.graph.mark_strip_edge(eid)
            affected.update(self.mesh.edge_to_tris[eid])
        for tri in affected:
            if self.graph.incident_strip_edge_count(tri) <= 1:
                self.terminals.add(tri)
            else:
                self.terminals.discard(tri)

# -------------------------------------------------------------------
# 6. Stats & Helpers
# -------------------------------------------------------------------

def compute_stats(sf: Stripifier) -> Tuple[int, int]:
    vis = set(); comps = []
    for t in range(sf.graph.num_triangles):
        if t not in vis:
            q=[t]; comp=[]; vis.add(t)
            while q:
                c=q.pop(); comp.append(c)
                for n,e in sf.graph.adj[c]:
                    if sf.graph.is_strip_edge[e] and n not in vis:
                        vis.add(n); q.append(n)
            comps.append(comp)
    return len(comps), sum(len(c) + 2 for c in comps)

# -------------------------------------------------------------------
# 7. Extract Strips with edges
# -------------------------------------------------------------------

def extract_strips(sf: Stripifier) -> List[Tuple[List[int], List[int]]]:
    vis = set(); strips = []
    # open strips
    for t in range(sf.graph.num_triangles):
        if sf.graph.incident_strip_edge_count(t) <= 1 and t not in vis:
            tris=[]; eds=[]; cur=t; prev=None
            while True:
                tris.append(cur); vis.add(cur)
                nexts = [(n,e) for n,e in sf.graph.adj[cur] if sf.graph.is_strip_edge[e] and n!=prev]
                if not nexts:
                    break
                prev, cur = cur, nexts[0][0]
                eds.append(nexts[0][1])
            strips.append((tris, eds))
    # cycles
    for t in range(sf.graph.num_triangles):
        if t not in vis and sf.graph.incident_strip_edge_count(t) == 2:
            tris=[t]; eds=[]; vis.add(t)
            nbs=[(n,e) for n,e in sf.graph.adj[t] if sf.graph.is_strip_edge[e]]
            prev, cur = t, nbs[0][0] if nbs else None
            eds.append(nbs[0][1] if nbs else None)
            while cur is not None and cur!=t:
                tris.append(cur); vis.add(cur)
                nxt=[(n,e) for n,e in sf.graph.adj[cur] if sf.graph.is_strip_edge[e] and n!=prev]
                if not nxt: break
                prev, cur = cur, nxt[0][0]
                eds.append(nxt[0][1])
            strips.append((tris, eds))
    return strips

# -------------------------------------------------------------------
# 8. Convert Strips back to vertices
# -------------------------------------------------------------------

def strips_to_vertex_indices(mesh: TriangleMesh, strips: List[Tuple[List[int], List[int]]]) -> List[List[int]]:
    """
    Converts strips (triangle ID lists and corresponding edge ID lists) into
    vertex index sequences for rendering.
    """
    vstrips: List[List[int]] = []
    for tris, eds in strips:
        if not tris:
            continue
        # Start with the first triangle's three vertices
        first_tri = mesh.triangles[tris[0]]
        seq = [first_tri.a, first_tri.b, first_tri.c]
        # Append one new vertex for each subsequent triangle
        for eid, tid in zip(eds, tris[1:]):
            # The edge gives the two shared vertices
            u, v = mesh.edge_to_vertices[eid]
            tri = mesh.triangles[tid]
            vids = {tri.a, tri.b, tri.c}
            # The new vertex is the one not in the shared edge
            new_vertex = (vids - {u, v}).pop()
            seq.append(new_vertex)
        vstrips.append(seq)
    return vstrips

In [81]:
tmesh = TriangleMesh(bunny_model.vertices, bunny_model.triangles)

In [12]:
# bunny_model

In [67]:
stripifier = Stripifier(tmesh)

# Stats before tunneling
before_strips, before_verts = compute_strip_stats(stripifier)
print(f"Before: {before_strips} strips, {before_verts} vertices")

# Run the stripifier
stripifier.run(max_iters=10000)

# Stats after tunneling
after_strips, after_verts = compute_strip_stats(stripifier)
print(f"After: {after_strips} strips, {after_verts} vertices")

# Active strip edges
# active_edges = [eid for eid, flag in stripifier.graph.is_strip_edge.items() if flag]
# print("Active strip edges:", active_edges)

Before: 4968 strips, 14904 vertices
After: 118 strips, 5204 vertices


In [68]:
17379 / 12

1448.25

In [69]:
len(tmesh.vertices)

2503

In [70]:
from utils.geometry import triangle_list_to_strip
triangle_strip = triangle_list_to_strip([[t.a, t.b, t.c] for t in tmesh.triangles])

In [71]:
len(triangle_strip)

11586

In [97]:
# verts=[Vertex(0,0,0),Vertex(1,0,0),Vertex(0,1,0),Vertex(1,1,0)]
# tris=[Triangle(0,1,2),Triangle(1,3,2)]
# mesh=TriangleMesh(verts,tris)
mesh=tmesh
stripifier=Stripifier(mesh)

b_s,b_v=compute_strip_stats(stripifier)
print(f"Before: {b_s} strips, {b_v} verts")
stripifier.run(max_iters=10000)
a_s,a_v=compute_strip_stats(stripifier)
print(f"After: {a_s} strips, {a_v} verts")

strips=extract_strips(stripifier)
vids=strips_to_vertex_indices(mesh,strips)
# Validate triangle coverage
ex_ids=[t for st in strips for t in st]
if set(ex_ids)!=set(range(len(mesh.triangles))):
    raise AssertionError("Triangle coverage mismatch")
if len(ex_ids)!=len(mesh.triangles):
    raise AssertionError("Duplicate triangle in strips")
# Validate connectivity
recon=[]
for seq in vids:
    for i in range(len(seq)-2):
        recon.append(tuple(sorted((seq[i],seq[i+1],seq[i+2]))))
orig=[tuple(sorted((t.a,t.b,t.c))) for t in mesh.triangles]
if sorted(recon)!=sorted(orig):
    raise AssertionError("Reconstructed triangles != original")
print("Validation passed.")

Before: 4968 strips, 14904 verts


TypeError: Stripifier.run() got an unexpected keyword argument 'max_iters'

In [56]:
strips

[[0, 1]]

In [57]:
vids

[[0, 1, 2, 3]]

In [58]:
recon

[(0, 1, 2), (1, 2, 3)]

In [46]:
orig

[(0, 1, 2), (1, 2, 3)]

In [98]:
verts = [Vertex(0,0,0), Vertex(1,0,0), Vertex(0,1,0), Vertex(1,1,0)]
tris = [Triangle(0,1,2), Triangle(1,3,2)]
mesh = TriangleMesh(verts, tris)
mesh = tmesh
sf = Stripifier(mesh)

b_s, b_v = compute_stats(sf)
print(f"Before: {b_s} strips, {b_v} verts")
sf.run(iters=5000)
a_s, a_v = compute_stats(sf)
print(f"After: {a_s} strips, {a_v} verts")

strips = extract_strips(sf)
vids = strips_to_vertex_indices (mesh, strips)

# Validate coverage
all_tris = [tid for tris, _ in strips for tid in tris]
assert set(all_tris) == set(range(len(mesh.triangles))), "Coverage error"
# Validate reconstruction
recon = []
for seq in vids:
    for i in range(len(seq)-2):
        recon.append(tuple(sorted((seq[i], seq[i+1], seq[i+2]))))
orig = [tuple(sorted((t.a,t.b,t.c))) for t in mesh.triangles]
assert sorted(recon) == sorted(orig), "Reconstruction error"
print("Validation passed.")

Before: 4968 strips, 14904 verts
After: 118 strips, 5204 verts


AssertionError: Reconstruction error

In [99]:
sorted(recon)

[(0, 201, 1164),
 (0, 419, 1064),
 (0, 1034, 1070),
 (0, 1064, 1065),
 (0, 1064, 1070),
 (0, 1065, 1164),
 (0, 1089, 1164),
 (0, 1089, 1205),
 (0, 1163, 1164),
 (1, 181, 1044),
 (1, 1033, 1044),
 (1, 1044, 1108),
 (1, 1044, 1299),
 (1, 1108, 1299),
 (2, 19, 346),
 (2, 19, 1088),
 (2, 315, 516),
 (2, 516, 892),
 (2, 516, 1088),
 (2, 892, 1831),
 (2, 892, 1843),
 (2, 1733, 1833),
 (2, 1831, 1833),
 (3, 48, 486),
 (3, 437, 457),
 (3, 437, 486),
 (4, 503, 1384),
 (4, 1356, 1504),
 (4, 1384, 1504),
 (4, 1384, 2157),
 (4, 1384, 2476),
 (4, 2015, 2157),
 (5, 125, 1154),
 (5, 196, 759),
 (5, 196, 1154),
 (5, 196, 1154),
 (5, 621, 1150),
 (5, 1150, 1154),
 (6, 116, 186),
 (6, 116, 1120),
 (6, 168, 1120),
 (7, 572, 852),
 (7, 641, 662),
 (7, 641, 758),
 (7, 641, 840),
 (7, 662, 663),
 (7, 840, 852),
 (8, 15, 730),
 (8, 15, 1037),
 (8, 150, 753),
 (8, 557, 730),
 (8, 557, 753),
 (8, 730, 1037),
 (9, 908, 979),
 (9, 908, 1395),
 (9, 979, 1016),
 (9, 979, 1450),
 (9, 1395, 1444),
 (9, 1450, 2166),


In [100]:
sorted(orig)

[(0, 201, 1089),
 (0, 201, 1164),
 (0, 1064, 1065),
 (0, 1064, 1070),
 (0, 1065, 1164),
 (0, 1070, 1089),
 (1, 181, 1044),
 (1, 181, 1108),
 (1, 432, 1044),
 (1, 432, 1299),
 (1, 1108, 1299),
 (2, 19, 1088),
 (2, 19, 1833),
 (2, 315, 516),
 (2, 315, 892),
 (2, 516, 1088),
 (2, 892, 1831),
 (2, 1831, 1833),
 (3, 167, 494),
 (3, 167, 495),
 (3, 437, 457),
 (3, 437, 485),
 (3, 457, 466),
 (3, 466, 495),
 (3, 485, 486),
 (3, 486, 494),
 (4, 1356, 1504),
 (4, 1356, 2157),
 (4, 1384, 1714),
 (4, 1384, 2157),
 (4, 1504, 1714),
 (5, 180, 196),
 (5, 180, 1154),
 (5, 196, 759),
 (5, 759, 1150),
 (5, 1150, 1154),
 (6, 116, 186),
 (6, 116, 1120),
 (6, 168, 390),
 (6, 168, 1120),
 (6, 186, 400),
 (6, 390, 667),
 (6, 400, 667),
 (7, 572, 662),
 (7, 572, 852),
 (7, 641, 662),
 (7, 641, 840),
 (7, 840, 852),
 (8, 33, 730),
 (8, 33, 1037),
 (8, 150, 753),
 (8, 150, 1037),
 (8, 557, 730),
 (8, 557, 753),
 (9, 302, 979),
 (9, 302, 1450),
 (9, 339, 1395),
 (9, 339, 2166),
 (9, 908, 979),
 (9, 908, 1395),


In [94]:
len(recon)

4968

In [95]:
len(orig)

4968

In [133]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import deque, namedtuple

# -------------------------------------------------------------------
# 1. Basic Primitives
# -------------------------------------------------------------------
@dataclass
class Vertex:
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0

@dataclass
class Triangle:
    a: int
    b: int
    c: int

# -------------------------------------------------------------------
# 2. Mesh Adjacency
# -------------------------------------------------------------------
@dataclass
class TriangleMesh:
    vertices: List[Vertex]
    triangles: List[Triangle]
    tri_to_edges: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_tris: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_vertices: Dict[int, Tuple[int, int]] = field(default_factory=dict)

    def build_adjacency(self) -> None:
        """
        Build:
        - tri_to_edges: tri index -> its edge IDs
        - edge_to_tris: edge ID -> incident tri indices
        - edge_to_vertices: edge ID -> vertex index pair
        """
        self.tri_to_edges.clear()
        self.edge_to_tris.clear()
        self.edge_to_vertices.clear()
        edge_map: Dict[Tuple[int, int], int] = {}
        next_eid = 0
        for tidx, tri in enumerate(self.triangles):
            eids: List[int] = []
            for u, v in ((tri.a, tri.b), (tri.b, tri.c), (tri.c, tri.a)):
                key = (min(u, v), max(u, v))
                if key not in edge_map:
                    edge_map[key] = next_eid
                    self.edge_to_tris[next_eid] = []
                    self.edge_to_vertices[next_eid] = key
                    next_eid += 1
                eid = edge_map[key]
                eids.append(eid)
                self.edge_to_tris[eid].append(tidx)
            self.tri_to_edges[tidx] = eids

# -------------------------------------------------------------------
# 3. Dual Graph
# -------------------------------------------------------------------
@dataclass
class DualGraph:
    num_triangles: int
    adj: Dict[int, List[Tuple[int, int]]] = field(default_factory=dict)
    is_strip_edge: Dict[int, bool] = field(default_factory=dict)

    @classmethod
    def from_mesh(cls, mesh: TriangleMesh) -> 'DualGraph':
        dg = cls(num_triangles=len(mesh.triangles))
        dg.adj = {t: [] for t in range(dg.num_triangles)}
        for eid, tris in mesh.edge_to_tris.items():
            dg.is_strip_edge[eid] = False
            # Connect every pair sharing this edge
            for i in range(len(tris)):
                for j in range(i + 1, len(tris)):
                    t0, t1 = tris[i], tris[j]
                    dg.adj[t0].append((t1, eid))
                    dg.adj[t1].append((t0, eid))
        return dg

    def mark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = True

    def unmark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = False

    def incident_strip_count(self, tri: int) -> int:
        return sum(self.is_strip_edge[eid] for _, eid in self.adj[tri])

# -------------------------------------------------------------------
# 4. Tunnel Finder
# -------------------------------------------------------------------
BFSState = namedtuple('BFSState', ['tri', 'parity', 'prev_eid', 'prev'])

class TunnelFinder:
    def __init__(self, graph: DualGraph):
        self.graph = graph

    def find_tunnel(self, start: int, max_depth: int = 100) -> Optional[List[int]]:
        visited = set()
        queue = deque([BFSState(start, 0, None, None)])
        while queue:
            state = queue.popleft()
            path = self._reconstruct(state)
            if len(path) > max_depth:
                continue
            for nbr, eid in self.graph.adj[state.tri]:
                want_strip = (state.parity == 1)
                if self.graph.is_strip_edge[eid] != want_strip:
                    continue
                nxt = BFSState(nbr, 1 - state.parity, eid, state)
                key = (nbr, nxt.parity)
                if key in visited:
                    continue
                visited.add(key)
                if nxt.parity == 1 and self._is_terminal(nbr) and nbr != start:
                    return self._reconstruct(nxt)
                queue.append(nxt)
        return None

    def _reconstruct(self, state: BFSState) -> List[int]:
        path = []
        while state.prev_eid is not None:
            path.append(state.prev_eid)
            state = state.prev
        return list(reversed(path))

    def _is_terminal(self, tri: int) -> bool:
        return self.graph.incident_strip_count(tri) <= 1

    # new
    def _is_free(self, tri: int) -> bool:
        return self.graph.incident_strip_count(tri) == 0

# -------------------------------------------------------------------
# 5. Stripifier
# -------------------------------------------------------------------
class Stripifier:
    def __init__(self, mesh: TriangleMesh):
        self.mesh = mesh
        self.mesh.build_adjacency()
        self.graph = DualGraph.from_mesh(mesh)
        self._reset_strips()
        self.tfinder = TunnelFinder(self.graph)
        self.terminals = {t for t in range(self.graph.num_triangles)
                          if self.graph.incident_strip_count(t) <= 1}

    def _reset_strips(self) -> None:
        for eid in self.graph.is_strip_edge:
            self.graph.unmark_strip(eid)

    def run(self, iters: int = 1000) -> None:
        for _ in range(iters):
            improved = False
            for t in list(self.terminals):
                path = self.tfinder.find_tunnel(t)
                if path:
                    self._apply(path)
                    improved = True
                    break
            if not improved:
                break

    def _apply(self, path: List[int]) -> None:
        affected = set()
        for eid in path:
            if self.graph.is_strip_edge[eid]:
                self.graph.unmark_strip(eid)
            else:
                self.graph.mark_strip(eid)
            affected.update(self.mesh.edge_to_tris[eid])
        for t in affected:
            if self.graph.incident_strip_count(t) <= 1:
                self.terminals.add(t)
            else:
                self.terminals.discard(t)

# -------------------------------------------------------------------
# 6. Helpers
# -------------------------------------------------------------------
def compute_stats(sf: Stripifier) -> Tuple[int, int, List[List[int]]]:
    visited = set(); comps = []
    for t in range(sf.graph.num_triangles):
        if t not in visited:
            stack = [t]; comp = []; visited.add(t)
            while stack:
                cur = stack.pop(); comp.append(cur)
                for nbr, eid in sf.graph.adj[cur]:
                    if sf.graph.is_strip_edge[eid] and nbr not in visited:
                        visited.add(nbr); stack.append(nbr)
            comps.append(comp)
    num = len(comps)
    verts = sum(len(c) + 2 for c in comps)
    return num, verts, comps

# -------------------------------------------------------------------
# 7. Extract & Convert
# -------------------------------------------------------------------
def extract_strips(sf: Stripifier) -> List[Tuple[List[int], List[int]]]:
    """
    Extract triangle-ID strips along with the connecting strip-edge IDs.
    Returns a list of (tris, eids) tuples:
      - tris: ordered triangle indices
      - eids: edge IDs connecting tris[i] to tris[i+1]
    """
    vis = set()
    strips: List[Tuple[List[int], List[int]]] = []
    # handle open strips
    for t in range(sf.graph.num_triangles):
        if sf.graph.incident_strip_count(t) <= 1 and t not in vis:
            tris = []
            eids = []
            cur, prev = t, None
            while True:
                tris.append(cur)
                vis.add(cur)
                # find next neighbor and edge
                nexts = [(n, eid) for n, eid in sf.graph.adj[cur]
                         if sf.graph.is_strip_edge[eid] and n != prev]
                if not nexts:
                    break
                prev, cur = cur, nexts[0][0]
                eids.append(nexts[0][1])
            strips.append((tris, eids))
    # handle cycles
    for t in range(sf.graph.num_triangles):
        if t not in vis and sf.graph.incident_strip_count(t) == 2:
            tris = [t]
            eids = []
            vis.add(t)
            neighs = [(n, eid) for n, eid in sf.graph.adj[t]
                      if sf.graph.is_strip_edge[eid]]
            if not neighs:
                continue
            prev, cur = t, neighs[0][0]
            eids.append(neighs[0][1])
            while cur is not None and cur != t:
                tris.append(cur)
                vis.add(cur)
                nexts = [(n, eid) for n, eid in sf.graph.adj[cur]
                         if sf.graph.is_strip_edge[eid] and n != prev]
                if not nexts:
                    break
                prev, cur = cur, nexts[0][0]
                eids.append(nexts[0][1])
            strips.append((tris, eids))
    return strips


def strips_to_vertex_indices(mesh: TriangleMesh, strips: List[Tuple[List[int], List[int]]]) -> List[List[int]]:
    """
    Convert triangle-ID strips and edge-ID lists to vertex index sequences.
    Uses edge_to_vertices to determine the two shared vertices for each transition.
    """
    vstrips: List[List[int]] = []
    for tris, eids in strips:
        if not tris:
            continue
        # first triangle
        tri0 = mesh.triangles[tris[0]]
        seq = [tri0.a, tri0.b, tri0.c]
        # for each connecting edge and next triangle
        for eid, tid in zip(eids, tris[1:]):
            u, v = mesh.edge_to_vertices[eid]
            tri = mesh.triangles[tid]
            vids = {tri.a, tri.b, tri.c}
            # the new vertex is the one not in the shared edge
            new_vid = (vids - {u, v}).pop()
            seq.append(new_vid)
        vstrips.append(seq)
    return vstrips

# -------------------------------------------------------------------
# -------------------------------------------------------------------
# 8. Example & Validation
# -------------------------------------------------------------------
# if __name__ == "__main__":
verts = [Vertex(0,0,0),Vertex(1,0,0),Vertex(0,1,0),Vertex(1,1,0)]
tris = [Triangle(0,1,2),Triangle(1,3,2)]
mesh = TriangleMesh(verts, tris)
mesh = tmesh
sf = Stripifier(mesh)

b_n, b_v, _ = compute_stats(sf)
print(f"Before: {b_n} strips, {b_v} verts")
sf.run(iters=5000)
a_n, a_v, _ = compute_stats(sf)
print(f"After: {a_n} strips, {a_v} verts")

strips = extract_strips(sf)  # List of (tris, eids)
vids = strips_to_vertex_indices(mesh, strips)

# validate coverage
covered = {t for tris, _ in strips for t in tris}
assert covered == set(range(len(mesh.triangles))), "Coverage mismatch"
# validate reconstruction
recon = [tuple(sorted((s[i],s[i+1],s[i+2]))) for s in vids for i in range(len(s)-2)]
orig = [tuple(sorted((t.a,t.b,t.c))) for t in mesh.triangles]
assert sorted(recon) == sorted(orig), "Reconstruction error"
print("Validation passed.")


Before: 4968 strips, 14904 verts
After: 118 strips, 5204 verts


AssertionError: Reconstruction error

In [134]:
compute_stats(sf)[2]

[[0, 1794, 168, 1982, 436, 3060, 1873, 3073, 463, 1882, 1796, 1881],
 [1,
  1870,
  1666,
  1836,
  3323,
  1868,
  3324,
  3259,
  3740,
  231,
  4286,
  3703,
  3700,
  3758,
  3757,
  3955,
  3258,
  3292,
  3282,
  3281,
  156,
  3307,
  3232,
  3233,
  3218,
  3199,
  3231,
  3245,
  3217,
  3230,
  3279,
  3305,
  1714,
  721,
  1681,
  3306,
  3280,
  2309,
  1754,
  1775,
  2364,
  37,
  722,
  2089,
  2085,
  1631,
  3927,
  4480,
  4487,
  4488,
  4489,
  4528,
  4441,
  4752,
  3030,
  2948,
  1720,
  1635,
  2945,
  2939,
  2946,
  3051,
  4563,
  4600,
  4592,
  4540,
  4541,
  4518,
  4495,
  4490,
  4470,
  4460,
  4469,
  4450,
  4468,
  4446,
  2078,
  1593,
  157],
 [2,
  34,
  4,
  3052,
  303,
  2308,
  590,
  571,
  559,
  2374,
  458,
  448,
  2399,
  524,
  2187,
  2186,
  2239,
  2136,
  2245,
  723,
  2307,
  2201,
  2202,
  2150,
  2172,
  2197,
  2372,
  2347,
  2211,
  2289,
  2129,
  2174,
  2184,
  280,
  224,
  207],
 [3,
  103,
  729,
  2243,
  2128,
  2

In [117]:
strips

[([4595,
   4582,
   4790,
   187,
   4520,
   4942,
   4919,
   3384,
   2961,
   4547,
   4918,
   4622,
   4433,
   4407,
   4438,
   4464,
   4747,
   4545,
   4931,
   4941,
   4800,
   4456,
   4859,
   4502,
   4809,
   4476,
   3494,
   4482,
   4930,
   4633,
   4808,
   4512,
   4937,
   4928,
   4906,
   4494,
   4892,
   4504,
   4503,
   4954,
   1740,
   4417,
   1663,
   1898,
   1930,
   1701,
   970,
   1884,
   1756,
   1770,
   1777,
   1818,
   1890,
   1707,
   1334,
   1857,
   1856,
   1409,
   2821,
   617,
   1917,
   1858,
   4140,
   4627,
   4894,
   1137,
   550,
   1286,
   4473,
   4443,
   4831,
   4425,
   4424,
   4461,
   4769,
   4774,
   4856,
   320,
   4626,
   4401,
   4554,
   4530,
   4927,
   4727,
   4726,
   4462,
   4463,
   4426,
   4749,
   2869,
   4467,
   4507,
   3355,
   4525,
   4434,
   4268,
   4459,
   4575,
   4395,
   4623,
   4759,
   3944,
   4630,
   4440,
   4399,
   4560,
   4689,
   4751,
   4631,
   4704,
   4688,
   461

In [113]:
sorted(recon)

[(0, 201, 1089),
 (0, 419, 1064),
 (0, 1064, 1065),
 (0, 1064, 1089),
 (0, 1065, 1164),
 (0, 1070, 1089),
 (0, 1089, 1164),
 (0, 1120, 1164),
 (0, 1163, 1164),
 (1, 181, 1044),
 (1, 181, 1108),
 (1, 432, 1033),
 (1, 432, 1044),
 (1, 432, 1299),
 (1, 1108, 1299),
 (1, 1299, 1380),
 (2, 19, 1088),
 (2, 19, 1833),
 (2, 315, 516),
 (2, 315, 892),
 (2, 516, 1088),
 (2, 892, 1831),
 (2, 1831, 1833),
 (3, 167, 494),
 (3, 167, 495),
 (3, 457, 466),
 (3, 466, 495),
 (3, 486, 494),
 (3, 486, 1125),
 (4, 1356, 1504),
 (4, 1356, 2157),
 (4, 1384, 1714),
 (4, 1384, 2157),
 (4, 1504, 1714),
 (5, 180, 196),
 (5, 180, 1154),
 (5, 196, 759),
 (5, 759, 1150),
 (5, 1150, 1154),
 (6, 116, 186),
 (6, 116, 1120),
 (6, 168, 1120),
 (7, 572, 662),
 (7, 572, 852),
 (7, 641, 662),
 (7, 641, 840),
 (7, 840, 852),
 (8, 106, 150),
 (8, 150, 1037),
 (8, 640, 753),
 (8, 696, 730),
 (8, 730, 753),
 (8, 730, 1037),
 (9, 302, 979),
 (9, 339, 1395),
 (9, 339, 2166),
 (9, 908, 979),
 (9, 908, 1395),
 (9, 1450, 2166),
 (1

In [114]:
sorted(orig)

[(0, 201, 1089),
 (0, 201, 1164),
 (0, 1064, 1065),
 (0, 1064, 1070),
 (0, 1065, 1164),
 (0, 1070, 1089),
 (1, 181, 1044),
 (1, 181, 1108),
 (1, 432, 1044),
 (1, 432, 1299),
 (1, 1108, 1299),
 (2, 19, 1088),
 (2, 19, 1833),
 (2, 315, 516),
 (2, 315, 892),
 (2, 516, 1088),
 (2, 892, 1831),
 (2, 1831, 1833),
 (3, 167, 494),
 (3, 167, 495),
 (3, 437, 457),
 (3, 437, 485),
 (3, 457, 466),
 (3, 466, 495),
 (3, 485, 486),
 (3, 486, 494),
 (4, 1356, 1504),
 (4, 1356, 2157),
 (4, 1384, 1714),
 (4, 1384, 2157),
 (4, 1504, 1714),
 (5, 180, 196),
 (5, 180, 1154),
 (5, 196, 759),
 (5, 759, 1150),
 (5, 1150, 1154),
 (6, 116, 186),
 (6, 116, 1120),
 (6, 168, 390),
 (6, 168, 1120),
 (6, 186, 400),
 (6, 390, 667),
 (6, 400, 667),
 (7, 572, 662),
 (7, 572, 852),
 (7, 641, 662),
 (7, 641, 840),
 (7, 840, 852),
 (8, 33, 730),
 (8, 33, 1037),
 (8, 150, 753),
 (8, 150, 1037),
 (8, 557, 730),
 (8, 557, 753),
 (9, 302, 979),
 (9, 302, 1450),
 (9, 339, 1395),
 (9, 339, 2166),
 (9, 908, 979),
 (9, 908, 1395),


In [130]:
[x[0] for x in strips if 0 in x[0]]

[[0, 1881, 1796, 1882, 463, 3073, 1873, 3060, 436, 1982, 168, 1794]]

In [121]:
2263 in strips[0]

False

In [123]:
type(strips[0])

tuple

In [126]:
2263 in strips[0][0]

False

In [127]:
strips[0][1]

[7072,
 7071,
 467,
 466,
 6999,
 7438,
 5616,
 4998,
 4997,
 7037,
 7117,
 6868,
 6830,
 6829,
 6875,
 6914,
 7033,
 7035,
 7449,
 7306,
 6901,
 6902,
 6971,
 6973,
 6931,
 5758,
 5757,
 6939,
 7129,
 7130,
 6988,
 6989,
 7447,
 7425,
 6960,
 6959,
 6976,
 6975,
 6974,
 3216,
 3217,
 3057,
 3056,
 3482,
 3138,
 1892,
 1891,
 3251,
 3250,
 3275,
 3291,
 3370,
 3150,
 2591,
 2592,
 3430,
 2715,
 2714,
 1304,
 1303,
 3432,
 3433,
 6555,
 7121,
 2229,
 1194,
 1193,
 2501,
 6887,
 6885,
 6857,
 6854,
 6855,
 6909,
 7281,
 7287,
 756,
 755,
 6817,
 6816,
 7010,
 7009,
 7237,
 7235,
 6911,
 6912,
 6858,
 6859,
 4850,
 4851,
 6919,
 5563,
 5565,
 6870,
 6688,
 6686,
 6906,
 6800,
 6799,
 7118,
 6322,
 6323,
 6879,
 6811,
 6810,
 7049,
 7202,
 7126,
 7125,
 7201,
 7104,
 7105,
 6819,
 6818,
 6814,
 6812,
 6871,
 7186,
 7221,
 6873,
 6837,
 6789,
 6788,
 6790,
 6794,
 6792,
 7262,
 6877,
 6832,
 6831,
 6860,
 3352,
 3351,
 7387,
 6953,
 6899,
 6898,
 6897,
 5891,
 5890,
 6796,
 6888,
 2536,
 25

In [145]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import deque, namedtuple

# -------------------------------------------------------------------
# 1. Basic Primitives
# -------------------------------------------------------------------
@dataclass
class Vertex:
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0

@dataclass
class Triangle:
    a: int
    b: int
    c: int

# -------------------------------------------------------------------
# 2. Mesh Adjacency
# -------------------------------------------------------------------
@dataclass
class TriangleMesh:
    vertices: List[Vertex]
    triangles: List[Triangle]
    tri_to_edges: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_tris: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_vertices: Dict[int, Tuple[int, int]] = field(default_factory=dict)

    def build_adjacency(self) -> None:
        self.tri_to_edges.clear()
        self.edge_to_tris.clear()
        self.edge_to_vertices.clear()
        edge_map: Dict[Tuple[int, int], int] = {}
        next_eid = 0
        for tidx, tri in enumerate(self.triangles):
            eids: List[int] = []
            for u, v in ((tri.a, tri.b), (tri.b, tri.c), (tri.c, tri.a)):
                key = (min(u, v), max(u, v))
                if key not in edge_map:
                    edge_map[key] = next_eid
                    self.edge_to_tris[next_eid] = []
                    self.edge_to_vertices[next_eid] = key
                    next_eid += 1
                eid = edge_map[key]
                eids.append(eid)
                self.edge_to_tris[eid].append(tidx)
            self.tri_to_edges[tidx] = eids

# -------------------------------------------------------------------
# 3. Dual Graph
# -------------------------------------------------------------------
@dataclass
class DualGraph:
    num_triangles: int
    adj: Dict[int, List[Tuple[int, int]]] = field(default_factory=dict)
    is_strip_edge: Dict[int, bool] = field(default_factory=dict)

    @classmethod
    def from_mesh(cls, mesh: TriangleMesh) -> 'DualGraph':
        dg = cls(num_triangles=len(mesh.triangles))
        dg.adj = {t: [] for t in range(dg.num_triangles)}
        for eid, tris in mesh.edge_to_tris.items():
            dg.is_strip_edge[eid] = False
            for i in range(len(tris)):
                for j in range(i + 1, len(tris)):
                    t0, t1 = tris[i], tris[j]
                    dg.adj[t0].append((t1, eid))
                    dg.adj[t1].append((t0, eid))
        return dg

    def mark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = True

    def unmark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = False

    def incident_strip_count(self, tri: int) -> int:
        return sum(self.is_strip_edge[eid] for _, eid in self.adj[tri])

# -------------------------------------------------------------------
# 4. Tunnel Finder
# -------------------------------------------------------------------
BFSState = namedtuple('BFSState', ['tri', 'parity', 'prev_eid', 'prev'])

class TunnelFinder:
    def __init__(self, graph: DualGraph):
        self.graph = graph

    def find_tunnel(self, start: int, max_depth: int = 100) -> Optional[List[int]]:
        # Only start from a free triangle
        if self.graph.incident_strip_count(start) != 0:
            return None
        visited = {(start, 0)}
        queue = deque([BFSState(start, 0, None, None)])
        while queue:
            state = queue.popleft()
            path = self._reconstruct(state)
            if len(path) > max_depth:
                continue
            for nbr, eid in self.graph.adj[state.tri]:
                want_strip = (state.parity == 1)
                if self.graph.is_strip_edge[eid] != want_strip:
                    continue
                nxt = BFSState(nbr, 1 - state.parity, eid, state)
                key = (nbr, nxt.parity)
                if key in visited:
                    continue
                visited.add(key)
                # Accept only if we reached a free triangle on a strip-edge step
                if nxt.parity == 1 and self.graph.incident_strip_count(nbr) == 0 and nbr != start:
                    return self._reconstruct(nxt)
                queue.append(nxt)
        return None

    def _reconstruct(self, state: BFSState) -> List[int]:
        path = []
        while state.prev_eid is not None:
            path.append(state.prev_eid)
            state = state.prev
        return list(reversed(path))

# -------------------------------------------------------------------
# 5. Stripifier
# -------------------------------------------------------------------
class Stripifier:
    def __init__(self, mesh: TriangleMesh):
        self.mesh = mesh
        self.mesh.build_adjacency()
        self.graph = DualGraph.from_mesh(mesh)
        self.tfinder = TunnelFinder(self.graph)
        # Triangles with zero incident strip-edges are free
        self.free_tris = {t for t in range(self.graph.num_triangles)
                          if self.graph.incident_strip_count(t) == 0}

    def run(self, iters: int = 1000) -> None:
        for _ in trange(iters):
            improved = False
            # Only try tunnels from free triangles
            for t in list(self.free_tris):
                path = self.tfinder.find_tunnel(t)
                if path:
                    self._apply(path)
                    improved = True
                    break
            if not improved:
                break

    def _apply(self, path: List[int]) -> None:
        # Toggle each edge along the augmenting path
        affected_tris = set()
        for eid in path:
            if self.graph.is_strip_edge[eid]:
                self.graph.unmark_strip(eid)
            else:
                self.graph.mark_strip(eid)
            affected_tris.update(self.mesh.edge_to_tris[eid])
        # Update free triangle set
        for t in affected_tris:
            if self.graph.incident_strip_count(t) == 0:
                self.free_tris.add(t)
            else:
                self.free_tris.discard(t)


In [137]:
verts = [Vertex(0,0,0),Vertex(1,0,0),Vertex(0,1,0),Vertex(1,1,0)]
tris = [Triangle(0,1,2),Triangle(1,3,2)]
mesh = TriangleMesh(verts, tris)
# mesh = tmesh
sf = Stripifier(mesh)

b_n, b_v, _ = compute_stats(sf)
print(f"Before: {b_n} strips, {b_v} verts")
sf.run(iters=5000)
a_n, a_v, _ = compute_stats(sf)
print(f"After: {a_n} strips, {a_v} verts")

strips = extract_strips(sf)  # List of (tris, eids)
vids = strips_to_vertex_indices(mesh, strips)

# validate coverage
covered = {t for tris, _ in strips for t in tris}
assert covered == set(range(len(mesh.triangles))), "Coverage mismatch"
# validate reconstruction
recon = [tuple(sorted((s[i],s[i+1],s[i+2]))) for s in vids for i in range(len(s)-2)]
orig = [tuple(sorted((t.a,t.b,t.c))) for t in mesh.triangles]
assert sorted(recon) == sorted(orig), "Reconstruction error"
print("Validation passed.")

Before: 2 strips, 6 verts
After: 1 strips, 4 verts
Validation passed.


In [148]:
# -------------------------------------------------------------------
# 6. Strip Extraction & Connectivity Check
# -------------------------------------------------------------------
def extract_triangle_strips(mesh: TriangleMesh, graph: DualGraph) -> List[List[int]]:
    visited = set()
    strips: List[List[int]] = []
    # start from all endpoints (deg <= 1)
    for t in range(graph.num_triangles):
        if t in visited:
            continue
        deg = graph.incident_strip_count(t)
        if deg > 1:
            continue
        # build strip
        strip = [t]
        visited.add(t)
        prev = None
        cur = t
        while True:
            # find next along matched edges excluding back edge
            moves = [(nbr, eid) for nbr, eid in graph.adj[cur]
                     if graph.is_strip_edge[eid] and nbr != prev]
            if len(moves) != 1:
                break
            nbr, eid = moves[0]
            if nbr in visited:
                break
            strip.append(nbr)
            visited.add(nbr)
            prev, cur = cur, nbr
        strips.append(strip)
    # any leftover triangles (in cycles or isolated) as length-1 strips
    for t in range(graph.num_triangles):
        if t not in visited:
            strips.append([t])
    return strips


def check_connectivity(mesh: TriangleMesh, strips: List[List[int]]) -> bool:
    original = {tuple(sorted((tri.a, tri.b, tri.c))) for tri in mesh.triangles}
    reconstructed = set()
    for strip in strips:
        for tidx in strip:
            tri = mesh.triangles[tidx]
            reconstructed.add(tuple(sorted((tri.a, tri.b, tri.c))))
    if original != reconstructed:
        missing = original - reconstructed
        extra = reconstructed - original
        print(f"Connectivity mismatch! Missing: {missing}, Extra: {extra}")
        return False
    print("Connectivity stays lossless: all triangles recovered.")
    return True

# -------------------------------------------------------------------
# 7. Vertex-Index Strip Extraction
# -------------------------------------------------------------------
def extract_vertex_strips(mesh: TriangleMesh, tri_strips: List[List[int]]) -> List[List[int]]:
    """
    Converts triangle-index strips into vertex-index strips for rendering.
    """
    vert_strips: List[List[int]] = []
    for strip in tri_strips:
        if not strip:
            continue
        # First triangle: output all three vertices
        first_tri = mesh.triangles[strip[0]]
        vs = [first_tri.a, first_tri.b, first_tri.c]
        # For each subsequent triangle, append the new vertex
        for tidx in strip[1:]:
            tri = mesh.triangles[tidx]
            # find the vertex not in the last two of vs
            last2 = set(vs[-2:])
            for v in (tri.a, tri.b, tri.c):
                if v not in last2:
                    vs.append(v)
                    break
        vert_strips.append(vs)
    return vert_strips

# -------------------------------------------------------------------
# 7. Example Usage
# -------------------------------------------------------------------
# if __name__ == "__main__":
# Build your mesh here
verts = [Vertex(0,0,0), Vertex(1,0,0), Vertex(0,1,0), Vertex(1,1,0)]
tris  = [Triangle(0,1,2), Triangle(1,3,2)]
# mesh = TriangleMesh(verts, tris)
mesh = TriangleMesh(bunny_model.vertices, bunny_model.triangles)
stripifier = Stripifier(mesh)
original_strips = extract_triangle_strips(mesh, stripifier.graph)
original_vert_strips = extract_vertex_strips(mesh, original_strips)
print(f"Before: {len(original_vert_strips)} strips, {sum(len(s) for s in original_vert_strips)} verts")
stripifier.run(iters=5000)
strips = extract_triangle_strips(mesh, stripifier.graph)
vert_strips = extract_vertex_strips(mesh, strips)
print(f"Before: {len(vert_strips)} strips, {sum(len(s) for s in vert_strips)} verts")
check_connectivity(mesh, strips)

Before: 4968 strips, 14904 verts


 50%|████▉     | 2484/5000 [00:00<00:00, 18383.25it/s]

Before: 2485 strips, 9938 verts
Connectivity stays lossless: all triangles recovered.


True

In [151]:
np.mean([len(s) for s in vert_strips])

np.float64(3.999195171026157)

In [174]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import deque, namedtuple

# -------------------------------------------------------------------
# 1. Basic Primitives
# -------------------------------------------------------------------
@dataclass
class Vertex:
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0

@dataclass
class Triangle:
    a: int
    b: int
    c: int

# -------------------------------------------------------------------
# 2. Mesh Adjacency
# -------------------------------------------------------------------
@dataclass
class TriangleMesh:
    vertices: List[Vertex]
    triangles: List[Triangle]
    tri_to_edges: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_tris: Dict[int, List[int]] = field(default_factory=dict)
    edge_to_vertices: Dict[int, Tuple[int, int]] = field(default_factory=dict)

    def build_adjacency(self) -> None:
        self.tri_to_edges.clear()
        self.edge_to_tris.clear()
        self.edge_to_vertices.clear()
        edge_map: Dict[Tuple[int, int], int] = {}
        next_eid = 0
        for tidx, tri in enumerate(self.triangles):
            eids: List[int] = []
            for u, v in ((tri.a, tri.b), (tri.b, tri.c), (tri.c, tri.a)):
                key = (min(u, v), max(u, v))
                if key not in edge_map:
                    edge_map[key] = next_eid
                    self.edge_to_tris[next_eid] = []
                    self.edge_to_vertices[next_eid] = key
                    next_eid += 1
                eid = edge_map[key]
                eids.append(eid)
                self.edge_to_tris[eid].append(tidx)
            self.tri_to_edges[tidx] = eids

# -------------------------------------------------------------------
# 3. Dual Graph
# -------------------------------------------------------------------
@dataclass
class DualGraph:
    num_triangles: int
    adj: Dict[int, List[Tuple[int, int]]] = field(default_factory=dict)
    is_strip_edge: Dict[int, bool] = field(default_factory=dict)

    @classmethod
    def from_mesh(cls, mesh: TriangleMesh) -> 'DualGraph':
        dg = cls(num_triangles=len(mesh.triangles))
        dg.adj = {t: [] for t in range(dg.num_triangles)}
        for eid, tris in mesh.edge_to_tris.items():
            dg.is_strip_edge[eid] = False
            for i in range(len(tris)):
                for j in range(i + 1, len(tris)):
                    t0, t1 = tris[i], tris[j]
                    dg.adj[t0].append((t1, eid))
                    dg.adj[t1].append((t0, eid))
        return dg

    def mark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = True

    def unmark_strip(self, eid: int) -> None:
        self.is_strip_edge[eid] = False

    def incident_strip_count(self, tri: int) -> int:
        return sum(self.is_strip_edge[eid] for _, eid in self.adj[tri])

# -------------------------------------------------------------------
# 4. Tunnel Finder (Augmenting Path Search)
# -------------------------------------------------------------------
BFSState = namedtuple('BFSState', ['tri', 'parity', 'prev_eid', 'prev'])

class TunnelFinder:
    def __init__(self, graph: DualGraph):
        self.graph = graph

    def find_tunnel(self, start: int, max_depth: int = 100) -> Optional[List[int]]:
        if self.graph.incident_strip_count(start) != 0:
            return None
        visited = {(start, 0)}
        queue = deque([BFSState(start, 0, None, None)])
        while queue:
            state = queue.popleft()
            if len(self._reconstruct(state)) > max_depth:
                continue
            for nbr, eid in self.graph.adj[state.tri]:
                want_strip = (state.parity == 1)
                if self.graph.is_strip_edge[eid] != want_strip:
                    continue
                nxt = BFSState(nbr, 1 - state.parity, eid, state)
                key = (nbr, nxt.parity)
                if key in visited:
                    continue
                visited.add(key)
                # if nxt.parity == 1 and self.graph.incident_strip_count(nbr) == 0 and nbr != start:
                if nxt.parity == 1 and self._is_terminal(nbr) and nbr != start:
                    return self._reconstruct(nxt)
                queue.append(nxt)
        return None

    def _reconstruct(self, state: BFSState) -> List[int]:
        path = []
        while state.prev_eid is not None:
            path.append(state.prev_eid)
            state = state.prev
        return list(reversed(path))

    def _is_terminal(self, tri: int) -> bool:
        return self.graph.incident_strip_count(tri) <= 1

# -------------------------------------------------------------------
# 5. Stripifier with Longest-Augmenting-Path Heuristic
# -------------------------------------------------------------------
class TunnelingStripifier:
    def __init__(self, mesh: TriangleMesh):
        self.mesh = mesh
        self.mesh.build_adjacency()
        self.graph = DualGraph.from_mesh(mesh)
        self._init()
        self.tfinder = TunnelFinder(self.graph)
        self.terminals = {t for t in range(self.graph.num_triangles)
                          if self.graph.incident_strip_count(t) <= 1}

    def _init(self) -> None:
        for eid in self.graph.is_strip_edge:
            self.graph.unmark_strip(eid)

    def run(self, iters: int = 1000) -> None:
        for _ in trange(iters):
            moved = False
            for st in list(self.terminals):
                tun = self.tfinder.find_tunnel(st)
                if tun:
                    self._apply(tun)
                    moved = True
                    break
            if not moved:
                break
    # def run(self, iters: int = 1000) -> None:
    #     for _ in range(iters):
    #         # Collect all augmenting paths from free triangles
    #         candidates: List[List[int]] = []
    #         free_tris = [t for t in range(self.graph.num_triangles)
    #                      if self.graph.incident_strip_count(t) == 0]
    #         for t in free_tris:
    #             path = self.tfinder.find_tunnel(t)
    #             if path:
    #                 candidates.append(path)
    #         if not candidates:
    #             break
    #         # Pick the longest augmenting path to maximize strip length
    #         best_path = max(candidates, key=len)
    #         self._apply(best_path)

    # def _apply(self, path: List[int]) -> None:
    #     # Toggle edges along the best path
    #     for eid in path:
    #         if self.graph.is_strip_edge[eid]:
    #             self.graph.unmark_strip(eid)
    #         else:
    #             self.graph.mark_strip(eid)

    def _apply(self, path: List[int]) -> None:
        affected = set()
        for eid in path:
            if self.graph.is_strip_edge[eid]:
                self.graph.unmark_strip(eid)
            else:
                self.graph.mark_strip(eid)
            affected.update(self.mesh.edge_to_tris[eid])
        for tri in affected:
            if self.graph.incident_strip_count(tri) <= 1:
                self.terminals.add(tri)
            else:
                self.terminals.discard(tri)
# -------------------------------------------------------------------
# 6. Strip Extraction & Connectivity Check
# -------------------------------------------------------------------
def extract_triangle_strips(mesh: TriangleMesh, graph: DualGraph) -> List[List[int]]:
    visited = set()
    strips: List[List[int]] = []
    for t in range(graph.num_triangles):
        if t in visited:
            continue
        deg = graph.incident_strip_count(t)
        if deg > 1:
            continue
        strip = [t]
        visited.add(t)
        prev = None
        cur = t
        while True:
            moves = [(nbr, eid) for nbr, eid in graph.adj[cur]
                     if graph.is_strip_edge[eid] and nbr != prev]
            if len(moves) != 1:
                break
            nbr, eid = moves[0]
            if nbr in visited:
                break
            strip.append(nbr)
            visited.add(nbr)
            prev, cur = cur, nbr
        strips.append(strip)
    for t in range(graph.num_triangles):
        if t not in visited:
            strips.append([t])
    return strips


def check_connectivity(mesh: TriangleMesh, strips: List[List[int]]) -> bool:
    original = {tuple(sorted((tri.a, tri.b, tri.c))) for tri in mesh.triangles}
    reconstructed = set()
    for strip in strips:
        for tidx in strip:
            tri = mesh.triangles[tidx]
            reconstructed.add(tuple(sorted((tri.a, tri.b, tri.c))))
    if original != reconstructed:
        missing = original - reconstructed
        extra = reconstructed - original
        print(f"Connectivity mismatch! Missing: {missing}, Extra: {extra}")
        return False
    print("Connectivity stays lossless: all triangles recovered.")
    return True

# -------------------------------------------------------------------
# 7. Vertex-Index Strip Extraction
# -------------------------------------------------------------------
def extract_vertex_strips(mesh: TriangleMesh, tri_strips: List[List[int]]) -> List[List[int]]:
    vert_strips: List[List[int]] = []
    for strip in tri_strips:
        if not strip:
            continue
        first_tri = mesh.triangles[strip[0]]
        vs = [first_tri.a, first_tri.b, first_tri.c]
        for tidx in strip[1:]:
            tri = mesh.triangles[tidx]
            last2 = set(vs[-2:])
            for v in (tri.a, tri.b, tri.c):
                if v not in last2:
                    vs.append(v)
                    break
        vert_strips.append(vs)
    return vert_strips

# -------------------------------------------------------------------
# 8. Example Usage
# -------------------------------------------------------------------
# if __name__ == "__main__":
# verts = [Vertex(0,0,0), Vertex(1,0,0), Vertex(0,1,0), Vertex(1,1,0)]
# tris  = [Triangle(0,1,2), Triangle(1,3,2)]

bunny_model = Reader.read_from_file('assets/stanford-bunny.obj')

mesh = TriangleMesh(bunny_model.vertices, bunny_model.triangles)
# mesh = tmesh
stripifier = Stripifier(mesh)
print(f"Before: {len(mesh.triangles)} strips, {len(mesh.triangles) * 3} verts")


stripifier.run(iters=10_000)
tri_strips = extract_triangle_strips(mesh, stripifier.graph)
# print("Triangle strips:", tri_strips)
vert_strips = extract_vertex_strips(mesh, tri_strips)
print(f"After: {len(vert_strips)} strips, {sum(len(s) for s in vert_strips)} verts")
# print("Vertex strips:", vert_strips)
check_connectivity(mesh, tri_strips)


Before: 69451 strips, 208353 verts


100%|██████████| 10000/10000 [00:20<00:00, 488.46it/s]


After: 59451 strips, 188353 verts
Connectivity stays lossless: all triangles recovered.


True

In [164]:
np.mean([len(s) for s in vert_strips])

np.float64(6.846829268292683)

In [166]:
np.max([len(s) for s in vert_strips])

np.int64(32)

In [167]:
np.min([len(s) for s in vert_strips])

np.int64(4)

In [165]:
len(bunny_model.triangles)

4968